In [12]:
from dotenv import load_dotenv
from langchain.agents import create_agent

load_dotenv()

True

Middleware，也就是中间件是一种控制Agent内部运行过程的技术，它在智能体运行的各个过程中预留钩子（hook），方便我们嵌入自定义操作。

基于Middleware可以实现各种高级功能，比如：
- 拦截和修改请求 - 在模型调用前后对输入输出进行处理
- 实现PII脱敏 - 自动检测和脱敏敏感个人信息
- 会话摘要管理 - 当对话过长时自动压缩历史消息
- 人工审核机制 - 在执行危险操作前等待人工确认
- 动态模型选择 - 根据运行时条件选择不同的模型
- 自定义状态管理 - 扩展Agent状态以跟踪额外信息


Middleware可以自己定义，也可以使用LangChain预定义好的。

# 1. 预定义中间件（Prebuilt Middleware）

我们会以其中的3个为例来介绍预定义中间件的用法：
- PIIMiddleware
- ModelFallbackMiddleware
- HumanInTheLoopMiddleware

## 1.1 PIIMiddleware - 个人信息脱敏
PIIMiddleware是一种预定义的wrap_model_call中间件，可以在调用模型前后自动检测并脱敏输入、输出消息中的个人身份信息（PII），如邮箱、电话号码、身份证号等。
其中，PII的脱敏处理策略有四种：
- 'block' - 抛出异常
- 'redact' - 用 [REDACTED_{PII_TYPE}] 来替代
- 'mask' - 关键信息采用**掩码 (例如., ****-****-****-1234)
- 'hash' - 用哈希值来替换

In [13]:
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import PIIMiddleware

# =================1.定义PIIMiddleware==========================
# 也可以添加多个PII中间件来检测不同类型的敏感信息
pii_middleware_email = PIIMiddleware(
    "email",
    strategy="redact",  # mask
    apply_to_input=False,
    apply_to_output=True
)

pii_middleware_phone = PIIMiddleware(
    "phone_number",
    # 使用自定义正则表达式
    detector=r"(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{4}",
    strategy="block",
    apply_to_input=True
)

# ====================2.在Agent中注册PIIMiddleware================
agent_with_builtin_middleware = create_agent(
    model="deepseek-v4-pro",
    middleware=[pii_middleware_email, pii_middleware_phone]
)

# ======================3.测试=================================
response = agent_with_builtin_middleware.invoke({
    "messages": [
        HumanMessage(
            "我的邮箱是: huge@itcast.cn,我的电话是13698023405,你要记住这些信息，后续要用到。如果记住了请确认一遍。")
    ]
})
for m in response['messages']:
    m.pretty_print()

PIIDetectionError: Detected 1 instance(s) of phone_number in text content

## 1.2 ModelFallbackMiddleware
ModelFallbackMiddleware的作用是在模型调用失败时给出降级处理方案。可以在创建时设置多个模型，如果主模型调用失败，会自动调用备用模型。

In [14]:
from langchain.agents.middleware import ModelFallbackMiddleware

# 1.Middleware，设置主模型和备用模型
model_fallback_middleware = ModelFallbackMiddleware(
    "gpt-4o-mini",  # 默认模型
    "deepseek-v4-pro"  #
)

# 2.创建Agent，设置Middleware
agent_with_model_fallback = create_agent(
    model="deepseek-chat",
    middleware=[model_fallback_middleware]
)

# 3.测试
response = agent_with_builtin_middleware.invoke({
    "messages": [HumanMessage("你好")]
})
for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================

你好呀！👋 很开心见到你！

我是DeepSeek，随时准备帮助你解决问题、聊天或者处理各种任务。无论你是想讨论问题、寻求建议、处理文件，还是只是想聊聊天，我都很乐意奉陪！

有什么我可以帮你的吗？😊


## 1.3 HumanInTheLoopMiddleware - 人工审核
HumanInTheLoop，简称为HITL，其作用是让人工介入到Agent执行流程中，在执行工具调用前暂停，等待人工确认。
通常用于Agent执行敏感操作前的确认，例如：
- 发送邮件
- 转账
- 执行脚本
- 读写文件
- ...

### 1.3.1 创建带有HITL的Agent
假如我们有一个负责转账操作的智能体，示例代码如下：
首先，定义一个转账的tool：

In [15]:
from langchain.tools import tool
# 1.定义工具
@tool
def transfer_money(amount: int, to: str):
    """向指定账户转账
    :arg
    amount: 金额
    to: 收款人
    """
    return f"已向账户{to}转账{amount}元"

然后，定义Middleware，设置需要人工确认的tool，以及人工确认的可选操作：

In [16]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

# 2.创建人工审核中间件，设定需要人工审核的 tool，以及审核的可选操作
human_in_loop_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "transfer_money": {
            "description": "请确认转账操作",
            "allowed_decisions": ["approve", "reject", "edit"]
        }
    }
)

代码解读：
- interrupt_on：就是需要人工确认的tool信息，可以设置多个
- transfer_money：就是需要人工确认的tool名字
- description：是人工确认时的提示信息
- allowed_decisions：是人工确认时的可选操作，包括三种：
  - approve：允许执行
  - reject：拒绝执行
  - edit：修改tool参数后执行

最后，创建智能体，设置Middleware：

In [18]:
from langgraph.checkpoint.memory import InMemorySaver
agent_with_hitl = create_agent(
    model="deepseek-chat",
    tools=[transfer_money],
    middleware=[human_in_loop_middleware],
    checkpointer=InMemorySaver(),
    system_prompt="你是一个账户管理助手，你可以调用工具帮助用户转账，用户提出明确需求后，不要重复向用户确认信息。"
)

接下来，测试一下：

In [19]:
config = {"configurable": {"thread_id": "3"}}
response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我转2000元给王小明(6123008415124395223)")]},
    config=config
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我转2000元给王小明(6123008415124395223)
================================== Ai Message ==================================

好的，我来帮您处理这笔转账，向王小明转账2000元。
Tool Calls:
  transfer_money (call_00_7Pnbn2Qc2XEEJFxEUkKZ3119)
 Call ID: call_00_7Pnbn2Qc2XEEJFxEUkKZ3119
  Args:
    amount: 2000
    to: 王小明


In [20]:
print(response)

{'messages': [HumanMessage(content='帮我转2000元给王小明(6123008415124395223)', additional_kwargs={}, response_metadata={}, id='c9723601-c7dd-4f59-93fa-83a2b16757c9'), AIMessage(content='好的，我来帮您处理这笔转账，向王小明转账2000元。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 340, 'total_tokens': 419, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 340}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '1ac549b2-8633-4ed4-8e4c-8e523a926f24', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f1233-68b5-7752-97e0-2faa737c5e83-0', tool_calls=[{'name': 'transfer_money', 'args': {'amount': 2000, 'to': '王小明'}, 'id': 'call_00_7Pnbn2Qc2XEEJFxEUkKZ3119', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 340

此时，如果Agent对接了前端，就应该在页面展示需要人工确认的提示信息：'请确认转账操作'，并给出approve、reject、edit这3个选项。
当用户选择一个操作后，将用户的选择告诉Agent。

那么问题来了，用户选择操作后，该如何将用户的选择告诉Agent呢？

### 1.3.2 approve
如果用户拒绝Agent调用工具，我们需要再次调用Agent，并通过Command来告知Agent用户的选择是什么。
示例：

In [21]:
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config  # Same thread ID to resume the paused conversation
)

print(response)

{'messages': [HumanMessage(content='帮我转2000元给王小明(6123008415124395223)', additional_kwargs={}, response_metadata={}, id='c9723601-c7dd-4f59-93fa-83a2b16757c9'), AIMessage(content='好的，我来帮您处理这笔转账，向王小明转账2000元。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 340, 'total_tokens': 419, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 340}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '1ac549b2-8633-4ed4-8e4c-8e523a926f24', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f1233-68b5-7752-97e0-2faa737c5e83-0', tool_calls=[{'name': 'transfer_money', 'args': {'amount': 2000, 'to': '王小明'}, 'id': 'call_00_7Pnbn2Qc2XEEJFxEUkKZ3119', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 340

1.3.3 reject
reject是拒绝执行tool，我们需要把拒绝的原因告知AI
示例：

In [22]:
response = agent_with_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # 用户reject的原因告知给ai
                    "message": "用户取消转账。"
                }
            ]
        }
    ),
    config=config
)

print(response)

{'messages': [HumanMessage(content='帮我转2000元给王小明(6123008415124395223)', additional_kwargs={}, response_metadata={}, id='c9723601-c7dd-4f59-93fa-83a2b16757c9'), AIMessage(content='好的，我来帮您处理这笔转账，向王小明转账2000元。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 340, 'total_tokens': 419, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 340}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '1ac549b2-8633-4ed4-8e4c-8e523a926f24', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f1233-68b5-7752-97e0-2faa737c5e83-0', tool_calls=[{'name': 'transfer_money', 'args': {'amount': 2000, 'to': '王小明'}, 'id': 'call_00_7Pnbn2Qc2XEEJFxEUkKZ3119', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 340

1.3.4 edit
edit是指需要人工对调用工具的参数做修改，所以要指定新的参数：

In [24]:
response = agent_with_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "transfer_money",
                        # Arguments to pass to the tool.
                        "args": {"amount": 1000, "to": "王小明"},
                    }
                }
            ]
        }
    ),
    config=config  # Same thread ID to resume the paused conversation
)

print(response)

{'messages': [HumanMessage(content='帮我转2000元给王小明(6123008415124395223)', additional_kwargs={}, response_metadata={}, id='c9723601-c7dd-4f59-93fa-83a2b16757c9'), AIMessage(content='好的，我来帮您处理这笔转账，向王小明转账2000元。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 340, 'total_tokens': 419, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 340}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '1ac549b2-8633-4ed4-8e4c-8e523a926f24', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f1233-68b5-7752-97e0-2faa737c5e83-0', tool_calls=[{'name': 'transfer_money', 'args': {'amount': 2000, 'to': '王小明'}, 'id': 'call_00_7Pnbn2Qc2XEEJFxEUkKZ3119', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 340

# 2. 自定义中间件
除了使用预置中间件，我们还可以利用LangChain提供的hook创建完全自定义的中间件。

根据hook的种类，中间件可以分为两类：
- Node-style hooks：在具体某个节点执行的中间件，包含：
  - before_agent
  - before_model
  - after_model
  - after_agent
- Wrap-style hooks：环绕model或tool调用的中间件，包括：
  - wrap_model_call
  - wrap_tool_call
为了便于开发中间件，LangChain为每一种hook都提供了装饰器，我们只有定义函数并使用装饰器标记即可快速开发中间件。

## 2.1 node-style装饰器
- Node-style hooks：在具体某个节点执行的中间件，包含：
  - before_agent
  - before_model
  - after_model
  - after_agent

例如，我们要实现模型调用计数功能，之前利用tool来统计增加了很多次tool调用，浪费token，而且也不太方便。利用中间件就优雅很多。我们可以在每次调用模型后(after_model)记录模型调用次数。

做法是定义函数，并用@after_model装饰器来装饰该函数，但要注意，函数的参数和返回值必须严格按照下面的示例：

In [25]:
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from typing import NotRequired, Any
from langchain.agents.middleware import after_model


# 1.定义自定义state，记录模型调用次数
class CustomAgentState(AgentState):
    """扩展Agent状态，添加自定义字段"""
    model_call_count: NotRequired[int]  # 模型调用次数


# 2.定义中间件
@after_model(state_schema=CustomAgentState)
def increment_counter(state: CustomAgentState, runtime: Runtime) -> dict[str, Any]:
    """使用装饰器的after_model钩子 - 增加调用计数"""
    current_count = state.get("model_call_count", 0)
    return {"model_call_count": current_count + 1}

测试：

In [27]:
# 创建智能体，设置middleware
agent = create_agent(
    model="deepseek-chat",
    middleware=[increment_counter],
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("Hello")],
    "model_call_count": 0,
}, config)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='31e50a89-63d6-4c1e-92e6-ace7908d06ff'), AIMessage(content='你好！很高兴见到你！😊\n\n有什么我可以帮你的吗？无论是回答问题、聊聊兴趣，还是需要协助处理什么事情，我都在这里！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 5, 'total_tokens': 36, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'acdae4ea-a8be-4265-895a-1908b820d18b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f123d-8341-7013-bb32-809c502fd7f1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 31, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})], 'model_call_count': 1

## 2.2 wrap-style装饰器
- Wrap-style hooks：环绕model或tool调用的中间件，包括：
  - wrap_model_call
  - wrap_tool_call

例如，我们来定义一个可以在调用模型时失败重试的中间件，最大重试次数为3次。那就可以使用@wrap_model_call，在模型调用时做出判断，如果失败则重试，重试此时超过3次则结束。

同样的，被@wrap_model_call装饰的函数，其参数和返回值必须严格按照下面的格式：

In [28]:
from langchain.agents.middleware import (
    wrap_model_call,
    ModelRequest,
    ModelResponse,
)
from typing import Any, Callable


@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request) # 这一步就是调用模型
        except Exception as e:
            print(f"Retry {attempt + 1}/3 after error: {e}")
            if attempt == 2:
                raise
    return None


agent = create_agent(
    model="deepseek-chat",
    middleware=[retry_model]
)

测试：

In [29]:
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {"messages": [HumanMessage("Hello")]},
    config,
)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='fc07fb54-d448-4454-8960-15d70e85247c'), AIMessage(content='你好！有什么可以帮你的吗？😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 5, 'total_tokens': 15, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '30d5b4cd-8964-4b46-af46-457ff3090c0a', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f123f-20ae-7a22-b5f6-1281e757cc59-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 10, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]}


## 2.3 类装饰器
对于需要同时用到多个hooks的更复杂的中间件逻辑，我们还可以使用自定义类继承AgentMiddleware的方式来创建中间件。
例如，一个用来记录日志的中间件，要在模型调用、工具调用前后记录日志：

In [30]:
from langchain_core.messages import AIMessage, ToolMessage
from langchain.agents.middleware import AgentMiddleware
from langchain.agents.middleware.types import ModelCallResult, ToolCallRequest
from langgraph.types import Command


class LoggingMiddleware(AgentMiddleware):

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelCallResult:
        try:
            print(f"\n=======About to call model with {len(request.messages)} messages=======")
            return handler(request)
        except Exception as e:
            print(f"\n=======[错误]: {str(e)}=======")
            return AIMessage("调用模型失败，请重试~")

    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
        print(f"\n=======调用工具: {request.tool_call['name']}=======")
        print(f"\n=======参数: {request.tool_call['args']}=======")
        try:
            result = handler(request)
            print("\n=======工具调用成功！=======")
            return result
        except Exception as e:
            print(f"\n=======工具调用失败: {e}=======")
            raise


@tool
def get_weather(location: str):
    """查询指定城市的天气信息"""
    return f"Current weather in {location} is sunny, 25℃."


agent = create_agent(
    model="deepseek-chat",
    middleware=[LoggingMiddleware()],
    tools=[get_weather],
)

for chunk, metadata in agent.stream(
    {"messages": [HumanMessage("杭州今天天气如何？")]},
    stream_mode="messages"
):
    if chunk and chunk.content:
        print(chunk.content, end="", flush=True)


=======About to call model with 1 messages=======
好的，我来查询一下杭州今天的天气情况。
=======调用工具: get_weather=======

=======参数: {'location': '杭州'}=======

=======工具调用成功！=======
Current weather in 杭州 is sunny, 25℃.
=======About to call model with 3 messages=======
杭州今天天气晴朗，☀️ 气温为 **25℃**，体感舒适，非常适合外出活动！出门的话注意适当防晒和补水哦～

# 3. 高级用法
中间件除了利用hook做基本的信息记录和判断，还可以有一些高级的用法，例如：
- 动态修改请求：可以拦截发送给模型的请求，动态修改请求中使用的模型、工具、提示词等
- 条件跳转：在满足条件的情况下直接跳转到某个Agent执行的节点


## 3.1 动态修改请求参数
在wrap_model_call这个hook中，我们可以动态修改任意的request参数，包括：
- model
- tool
- system_prompt
- ...
关键点是利用request.override()来覆盖原本的request参数。

例如，我们定义一个用来动态切换model的中间件，在每次模型调用前判断是采用推理模型还是普通模型。

In [31]:
# 使用 @wrap_model_call 装饰器创建中间件
# 这种方式适合简单的请求修改
from langchain.agents.middleware import wrap_model_call
from collections.abc import Callable
from pydantic.dataclasses import dataclass
from langchain.chat_models import init_chat_model

# 初始化不同的模型
reasoning_model = init_chat_model(model="deepseek-reasoner")
chat_model = init_chat_model(model="deepseek-chat")


# 定义一个context，记录用户是否使用推理模型
@dataclass
class UserContext:
    reasoning: bool = False


@wrap_model_call
def dynamic_model_selector(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """
    根据context中的reasoning来选择模型：
    - True: 选择推理模型
    - False: 选择普通对话模型
    """
    # 获取上下文参数
    is_reasoning = getattr(request.runtime.context, 'reasoning', False)

    # 根据消息数量选择模型
    selected_model = reasoning_model if is_reasoning else chat_model
    print(f"选择模型: {selected_model.model_name} (reasoning={is_reasoning})")

    # 覆盖request中的model参数
    modified_request = request.override(model=selected_model)
    return handler(modified_request)

测试：

In [32]:
# 创建智能体，设置middleware
agent = create_agent(
    model="deepseek-chat",
    middleware=[dynamic_model_selector],
    checkpointer=InMemorySaver(),
    context_schema=UserContext
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {"messages": [HumanMessage("Hello")]},
    config,
    context=UserContext(reasoning=True) # 通过传递Context控制模型的类型
)

print(response)

选择模型: deepseek-reasoner (reasoning=True)
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='ed8b03dd-9a28-4655-9e9b-0460011980ce'), AIMessage(content='Hello! How can I help you today? I’m DeepSeek—whether you need information, help with a task, or just want to chat, I’m here for you.', additional_kwargs={'refusal': None, 'reasoning_content': 'Okay, the user just said "Hello". This is a very simple and common greeting. There\'s no specific question or request attached. My primary goal is to be friendly and open the conversation in a helpful way. \n\nSince the user is a new user, the best approach is to warmly acknowledge the greeting and then invite further interaction. I should introduce myself briefly and offer assistance. This signals that I\'m ready to help with anything—whether they have a specific task, want to chat, or need information.\n\nI can keep it simple: return the greeting, state my name, ask how I can help, and maybe suggest a cou

## 3.2 条件跳转
在中间件使用jump_to指令可以跳过模型调用，直接进入指定的Agent节点。

例如，我们设定一个模型限流的中间件，当判断用户模型调用次数超过阈值，直接禁止调用，结束Agent运行。

In [33]:
from langgraph.runtime import Runtime
from typing import NotRequired, Any
from langchain.agents.middleware import AgentMiddleware, hook_config, AgentState


# 1.定义自定义state，记录模型调用次数
class CustomAgentState(AgentState):
    """扩展Agent状态，添加自定义字段"""
    model_call_count: NotRequired[int]  # 模型调用次数


# 2.定义中间件
class ModelCallLimitMiddleware(AgentMiddleware[CustomAgentState]):

    def __init__(self, max_limit: int = 10):
        super().__init__()
        self.max_limit = max_limit

    # after_model hook，模型调用次数计数
    def after_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        current_count = state.get("model_call_count", 0)
        return {"model_call_count": current_count + 1}

    # before_model hook，校验模型调用次数是否超限
    @hook_config(can_jump_to=["end"])
    def before_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        # 获取模型调用次数
        current_count = state.get("model_call_count", 0) + 1
        # 判断是否超限
        print(f"当前模型调用次数：{current_count}/{self.max_limit}")
        if current_count > self.max_limit:
            return {
                "jump_to": "end",
                "messages": AIMessage(f"模型调用次数超过最大限制：{self.max_limit}！")
            }
        return None

# @before_model(can_jump_to=["end"])
# def check_call_limit(state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
#     pass

测试：

In [35]:
# 创建智能体，设置middleware
agent = create_agent(
    model="deepseek-chat",
    middleware=[ModelCallLimitMiddleware(max_limit=2)],
    checkpointer=InMemorySaver(),
    state_schema=CustomAgentState
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("Hello")],
    "model_call_count": 0,
}, config)

print(response)

当前模型调用次数：1/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='3ff2e364-a775-432d-99b8-49e1a1cc58ec'), AIMessage(content='你好！有什么可以帮你的吗？😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 5, 'total_tokens': 15, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '2cd3dd1b-e9e6-416e-b286-15436ac8a9b7', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f1245-698b-7b02-90c2-4a6650936b2b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 10, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})], 'model_call_count': 1}
